### Multimodal RAG (PDF with Images)

![pdf-libraries comparision.png](../assets/pdf-libraries%20comparision.png)

In [2]:
import pymupdf # PyMuPDF
from langchain_core.documents import Document
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
import torch
import numpy as np
from langchain.chat_models import init_chat_model
from langchain_core.prompts import PromptTemplate
from langchain_core.messages import HumanMessage
from sklearn.metrics.pairwise import cosine_similarity
import os
import base64
import io
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS

In [3]:
###  Clip Model
from dotenv import load_dotenv

load_dotenv()

## Set up the environment
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

### Intialize the Clip Model for unified embeddings
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
clip_model.eval()

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 5045.91it/s]


CLIPModel(
  (text_model): CLIPTextModel(
    (embeddings): CLIPTextEmbeddings(
      (token_embedding): Embedding(49408, 512)
      (position_embedding): Embedding(77, 512)
    )
    (encoder): CLIPEncoder(
      (layers): ModuleList(
        (0-11): 12 x CLIPEncoderLayer(
          (self_attn): CLIPAttention(
            (k_proj): Linear(in_features=512, out_features=512, bias=True)
            (v_proj): Linear(in_features=512, out_features=512, bias=True)
            (q_proj): Linear(in_features=512, out_features=512, bias=True)
            (out_proj): Linear(in_features=512, out_features=512, bias=True)
          )
          (layer_norm1): LayerNorm((512,), eps=1e-05, elementwise_affine=True, bias=True)
          (mlp): CLIPMLP(
            (activation_fn): QuickGELUActivation()
            (fc1): Linear(in_features=512, out_features=2048, bias=True)
            (fc2): Linear(in_features=2048, out_features=512, bias=True)
          )
          (layer_norm2): LayerNorm((512,), eps=1

In [4]:
### Embedding functions
def embed_image(image_data):
  """Embed image using CLIP"""
  if isinstance(image_data, str): # If path
    image = Image.open(image_data).convert("RGB")
  else: # If PIL Image
    image = image_data

  inputs = clip_processor(images=image, return_tensors="pt")
  with torch.no_grad():
    features = clip_model.get_image_features(**inputs)
    # Check if the output is not a raw Tensor (for transformers v5.0+)
    if not isinstance(features, torch.Tensor):
      features = features.pooler_output
    # Normalize embeddings to unit vector
    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()

def embed_text(text):
  """Embed text using CLIP."""
  inputs = clip_processor(
    text=text,
    return_tensors="pt",
    padding=True,
    truncation=True,
    max_length=77 # Clip max token length
  )

  with torch.no_grad():
    features = clip_model.get_text_features(**inputs)
    # Check if the output is not a raw Tensor (for transformers v5.0+)
    if not isinstance(features, torch.Tensor):
      features = features.pooler_output
    # Normalize embeddings
    features = features / features.norm(dim=-1, keepdim=True)
    return features.squeeze().numpy()


In [5]:
## Process PDF
pdf_path = "multimodal_sample.pdf"
doc = pymupdf.open(pdf_path)

# Storage for all documents and embeddings
all_docs = []
all_embeddings = []
image_data_store = {} # Store actual image data for LLM

# Text Splitter
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=100)

In [6]:
for i, page in enumerate(doc):
  ## Process the text
  text = page.get_text()
  if text.strip():
    ## Create temporary document for splittng
    temp_doc = Document(page_content=text, metadata={"page": i, "type": "text"})
    text_chunks = splitter.split_documents([temp_doc])

    # Embed each chunk using clip
    for chunk in text_chunks:
      embedding = embed_text(chunk.page_content)
      all_embeddings.append(embedding)
      all_docs.append(chunk)
  
  
  ## Process images
  """
    Three Important Actions:

    # Convert PDF Image to PIL format
    # Store as base64 for GPT-4V (which needs base64 images)
    # Create CLIP embbding for retrival
  """
  for img_index, img in enumerate(page.get_images(full=True)):
    try:
      xref = img[0]
      base_image = doc.extract_image(xref)
      image_bytes = base_image["image"]

      # Convert to PIL Image
      pil_image = Image.open(io.BytesIO(image_bytes)).convert("RGB")

      # Create unique identifier
      image_id = f"page_{i}_img_{img_index}"

      # Store image as base64 for later use with GPT-4V
      buffered = io.BytesIO()
      pil_image.save(buffered, format="PNG")
      img_base64 = base64.b64encode(buffered.getvalue()).decode()
      image_data_store[image_id] = img_base64

      # Embed image using CLIP
      embedding = embed_image(pil_image)
      all_embeddings.append(embedding)

      # Create document for image
      image_doc = Document(
        page_content=f"[Image: {image_id}]",
        metadata = {"page": i, "type": "image", "image_id": image_id}
      )
      all_docs.append(image_doc)
    except Exception as e:
      print(f"Error processing image {img_index} on page {i}: {e}")
      continue

doc.close()

In [7]:
all_embeddings

[array([-2.67243409e-03,  1.28300032e-02, -5.18314578e-02,  4.14879769e-02,
        -2.33942121e-02, -7.55866384e-03, -3.67658958e-02,  1.19710758e-01,
         8.52081031e-02,  2.05425802e-03, -1.11534679e-02, -1.29592493e-02,
         5.25014699e-02, -3.65395634e-03,  4.76078689e-02,  1.58373173e-02,
         2.03388166e-02,  4.35361676e-02, -3.29166697e-03,  2.03181021e-02,
         1.88026356e-03, -4.23493907e-02,  5.44106588e-03,  3.70935760e-02,
        -1.65622681e-02,  6.48646150e-03, -4.78012525e-02,  8.67483206e-03,
         5.88859916e-02, -3.21394168e-02,  4.32440042e-02,  9.65301506e-03,
        -4.47922247e-03, -1.94857791e-02, -3.63502838e-02, -1.23471869e-02,
        -2.17929259e-02, -1.99016389e-02,  8.09620097e-02, -3.32986899e-02,
        -2.38901302e-02, -3.96139100e-02, -1.27280215e-02,  3.50380950e-02,
        -2.52216924e-02,  2.00031349e-03,  1.49660297e-02, -2.31976490e-02,
        -6.86790869e-02, -5.25782583e-04, -2.22545825e-02, -1.04103833e-02,
        -1.9

In [8]:
all_docs

[Document(metadata={'page': 0, 'type': 'text'}, page_content='Annual Revenue Overview\nThis document summarizes the revenue trends across Q1, Q2, and Q3. As illustrated in the chart\nbelow, revenue grew steadily with the highest growth recorded in Q3.\nQ1 showed a moderate increase in revenue as new product lines were introduced. Q2 outperformed\nQ1 due to marketing campaigns. Q3 had exponential growth due to global expansion.'),
 Document(metadata={'page': 0, 'type': 'image', 'image_id': 'page_0_img_0'}, page_content='[Image: page_0_img_0]')]

In [9]:
embeddings_array = np.array(all_embeddings)
embeddings_array

array([[-0.00267243,  0.01283   , -0.05183146, ..., -0.00385087,
         0.02977719, -0.00010682],
       [ 0.01732337, -0.0132769 , -0.0242703 , ...,  0.08994053,
        -0.00272156,  0.03253038]], shape=(2, 512), dtype=float32)

In [10]:
# Create unified FAISS vector store with CLIP embeddings


# Create custom FAISS index since we have precomputed embeddings
vector_store = FAISS.from_embeddings(
  text_embeddings=[(doc.page_content, emb) for doc, emb in zip(all_docs, embeddings_array)],
  embedding=None, # We are using precomputed embeddings
  metadatas=[doc.metadata for doc in all_docs]
)

`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


In [11]:
# Intialize GPT-4 Vision model
llm = init_chat_model("openai:gpt-4.1")
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.5.3', 'langchain': '1.3.14', 'langchain-openai': '1.4.3'}}, output_version=None, profile={'name': 'GPT-4.1', 'release_date': '2025-04-14', 'last_updated': '2025-04-14', 'open_weights': False, 'max_input_tokens': 1047576, 'max_output_tokens': 32768, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': True, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x00000207A06DA120>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x00000207A06DABA0>, root_client=<openai.OpenAI object at 

In [12]:
def retrieve_multimodal(query, k=5):
  """Unified retrieval using CLIP embeddings for both text and images."""
  # Embed query using CLIP
  query_embedding = embed_text(query)

  # Search in unified vector store
  results = vector_store.similarity_search_by_vector(
    embedding=query_embedding,
    k=k
  )

  return results

In [13]:
def create_multimodal_message(query, retrieved_docs):
  """Create a message with both text and images for GPT-4V."""
  content=[]

  # Add the query
  content.append({
    "type": "text",
    "text": f"Question: {query}\n\nContext:\n"
  })

  # Seperate text and images documents
  text_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "text"]
  image_docs = [doc for doc in retrieved_docs if doc.metadata.get("type") == "image"]

  # Add text context
  if text_docs:
    text_context = "\n\n".join([
      f"[Page {doc.metadata['page']}]: {doc.page_content}"
      for doc in text_docs
    ])

    content.append({
      "type": "text",
      "text": f"Text excerpts: \n{text_context}\n"
    })

  # Add images
  for doc in image_docs:
    image_id = doc.metadata.get("image_id")
    if image_id and image_id in image_data_store:
      content.append({
        "type": "image_url",
        "image_url": {
          "url": f"data:image/png;nase64,{image_data_store[image_id]}"
        }
      })

  # Add instruction
  content.append({
    "type": "text",
    "text": "\n\nPlease answer the question based on the provided text and images."
  })

  return HumanMessage(content=content)

In [14]:
def multimodal_pdf_rag_pipeline(query):
  """Main pipeline for multimodal RAG"""

  # Retrieve relevant documents
  context_docs = retrieve_multimodal(query, k=5)

  # Create multimodal message
  message = create_multimodal_message(query, context_docs)

  # Get response from GPT-4V
  response = llm.invoke([message])

  # Print received context info
  print(f"\nRetrieved {len(context_docs)} documents:")
  for doc in context_docs:
    doc_type = doc.metadata.get("type", "unknown")
    page = doc.metadata.get("page", "?")
    if doc_type == "text":
      preview = doc.page_content[:100] + "..." if len(doc.page_content) > 100 else doc.page_content
      print(f" - Text from page {page} : {preview}")
    else:
      print(f" - Image from page {page}")
    print("\n")

    return response.content

In [15]:
if __name__ == "__main__":
  # Example queries

  queries = [
    "What does the chart on page 1 show about revenue trends?",
    "Summarize the main findings from the document",
    "What visual elements are present in the document?"
  ]

  for query in queries:
    print(f"\nQuery: {query}")
    print("-"*50)
    answer = multimodal_pdf_rag_pipeline(query)
    print(f"Answer: {answer}")
    print("="*70)


Query: What does the chart on page 1 show about revenue trends?
--------------------------------------------------


RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}